In [1]:
# ================================================================
#  LCCN-R17 - MERGE-PRECISION AUDIT ON A UNIFORM SAMPLE  [CPU, ~20 min]
#  Answers IJIES Reviewer 1, point 7 and supports Reviewer 2, point 1.
#  Table 5 precision is computed on labelled pairs only (both forms carry
#  one emotion), a subset whose share varies by method. Cells 0-2 are the
#  certified setup (pipeline v5.3, lexicon, F2E, frozen split).
#  PHASE 1 (runs now)
#    Cell 3  rebuilds the merged-pair buckets of every merging method from
#            the training split exactly as paper1-lccn-merge does, draws
#            200 merged pairs per method UNIFORMLY over all merged pairs,
#            attaches two contexts per form, shuffles, and writes a BLIND
#            sheet merge_audit_sheet.csv (method hidden; key in
#            merge_audit_key.csv)
#    Cell 4  coverage column (labelled / merged) and the Table 5 precision
#            and correlation reproduced for the ten merging methods
#  PHASE 2 (after annotation): attach merge_audit_done.csv with ann1 and
#    ann2 filled (C = compatible: the two forms are spellings of the same
#    word or carry the same emotion; X = incompatible; U = unclear) and
#    re-run: Cell 5 scores kappa, audited precision per method, agreement
#    with the emotion-based X(f), and the correlation with mean macro-F1.
#  Add Input: paper1-nb1-lccn-final (output), final-emotion-dataset,
#             gold-ds, and in Phase 2 the dataset with merge_audit_done.csv.
#  Accelerator None. Internet ON. Save & Run All (Commit).
# ================================================================
import subprocess, sys
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q regex emoji wordfreq indic-nlp-library')
print("python", sys.version.split()[0])


+ pip install -q regex emoji wordfreq indic-nlp-library
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 3.1 MB/s eta 0:00:00
python 3.13.15


In [2]:
# ================================================================
#  CELL 1 - CERTIFIED PIPELINE (v5.3, selected explicitly)
# ================================================================
import shutil, importlib, sys, glob, re, os
for p in ("/kaggle/working/pipeline.py",):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("/kaggle/working/__pycache__", ignore_errors=True)
sys.modules.pop("pipeline", None)
cand = sorted(glob.glob("/kaggle/input/**/pipeline.py", recursive=True))
assert cand, "attach the paper1-nb1-lccn-final output"
chosen = None
for p in cand:
    m = re.search(r'PIPELINE_VERSION\s*=\s*"([^"]+)"',
                  open(p, encoding="utf-8", errors="replace").read())
    v = m.group(1) if m else "?"
    print(f"  v{v:<5} {p}")
    if v == "5.3" and chosen is None: chosen = p
assert chosen, "no v5.3 pipeline.py among the inputs"
shutil.copy(chosen, "/kaggle/working/pipeline.py")
if "/kaggle/working" not in sys.path: sys.path.insert(0, "/kaggle/working")
import pipeline
FP = pipeline.fingerprint()
assert pipeline.PIPELINE_VERSION == "5.3" and FP == "6a536dd437ba85b0"


  v5.3   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/pipeline.py
pipeline v5.3  sha256:6a536dd437ba85b0  SEGMENT=True


In [3]:
# ================================================================
#  CELL 2 - SHARED SETUP (replicates NB1 CELL 2; frozen split asserted)
# ================================================================
import importlib
importlib.reload(pipeline)
from pipeline import *
import pandas as pd, numpy as np
from collections import Counter

df, ID, TXT, LAB, RATERS = load_corpus()
norm_map, F2E, LEX = build_keel()
try:
    from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    _n = IndicNormalizerFactory().get_normalizer("kn")
    _n.normalize("\u0c95"); pipeline._indic_norm = _n; print("IndicNLP: FULL")
except Exception:
    import unicodedata
    class _F:
        def normalize(self, s): return unicodedata.normalize("NFC", s)
    pipeline._indic_norm = _F(); print("IndicNLP: FALLBACK(NFC)")

df["text_raw"] = df[TXT].map(lambda t: " ".join(process_comment(t)[0]))
SPLIT, TEST_HASH = make_split(df, ID, LAB)
assert TEST_HASH == "e2d6e1f7d564cace", f"split drift: {TEST_HASH}"
train = df[df[ID].isin(SPLIT["train"])]
evidence = LabelEvidence(train, TXT, LAB)
TRAIN_FREQ = Counter()
for t in train.text_raw: TRAIN_FREQ.update(w.lower() for w in t.split())
pipeline.TRAIN_FREQ = TRAIN_FREQ
print(f"train vocab {len(TRAIN_FREQ):,} | hash {TEST_HASH}")


corpus: 40,000 -> 40,000 after audit (dedup + conflict removal)
majority
disgust    10000
happy      10000
sad        10000
anger      10000
  KEEL cascade | tier1 126,567 | tier2 0 | tier3 603 | emotions 174,425 (3,658 ambiguous)
    multi-word entries skipped (word-level lookup): 274,777
    negated phrases kept for the context-aware veto: 706 | va_id misses 0
  KEEL entries: 126,567 | seg vocab: 12,326 | form->emotion: 174,425
IndicNLP: FULL
train  28,000
val     6,000
test    6,000
leakage check passed | test-set id hash: e2d6e1f7d564cace
train vocab 42,985 | hash e2d6e1f7d564cace


In [4]:
# ================================================================
#  CELL 3 - BUCKETS, UNIFORM SAMPLE OF 200 PAIRS PER METHOD, BLIND SHEET
# ================================================================
import random, glob, os
from collections import defaultdict, Counter
from scipy.stats import pearsonr, spearmanr
random.seed(42); np.random.seed(42)
sd = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
train = pd.read_csv(f"{sd[0]}/train.csv", dtype=str, keep_default_na=False)
train.columns = [c.strip().lower() for c in train.columns]; assert len(train) == 28000
METHODS = ["keel", "kope", "metaphone", "terun", "soundex", "soundex_guard",
           "indicsoundex", "indicsoundex_guard", "lccn_base", "lccn"]
for m in METHODS: assert f"text_{m}" in train.columns, m
single = {f: next(iter(e)) for f, e in F2E.items() if len(e) == 1}
print(f"single-emotion forms {len(single):,}")
ctx = defaultdict(list)
for raw in train["text_raw"]:
    toks = raw.split()
    for i, t in enumerate(toks):
        w = t.lower()
        if len(ctx[w]) < 4: ctx[w].append(" ".join(toks[max(0, i-4):i+5]))
def buckets_for(m):
    mp = defaultdict(Counter)
    for a, b in zip(train["text_raw"], train[f"text_{m}"]):
        ra, rb = a.split(), b.split()
        if len(ra) != len(rb): continue
        for w, x in zip(ra, rb): mp[w.lower()][x.lower()] += 1
    out = {f: c.most_common(1)[0][0] for f, c in mp.items()}
    bk = defaultdict(list)
    for f, o in out.items(): bk[o].append(f)
    return {o: sorted(fs) for o, fs in bk.items() if len(fs) >= 2}

stats = {}; sheet = []; key = []
for mi, m in enumerate(METHODS):
    bk = buckets_for(m)
    outs = list(bk); npairs = np.array([len(bk[o]) * (len(bk[o]) - 1) // 2 for o in outs], dtype=float)
    total = int(npairs.sum())
    # labelled pairs and X(f), Table 5 definition
    lab = same = 0
    for fs in bk.values():
        ce = Counter(single[f] for f in fs if f in single); L = sum(ce.values())
        lab += L*(L-1)//2; same += sum(k*(k-1)//2 for k in ce.values())
    x = lab - same; prec = 1 - x/lab if lab else float("nan")
    stats[m] = dict(method=m, merged_pairs=total, labelled_pairs=lab, coverage_pct=round(100*lab/total, 2) if total else float("nan"),
                    x_cross=x, precision_labelled=round(prec, 4))
    # uniform sample over pairs
    seen = set(); tries = 0
    while len(seen) < min(200, total) and tries < 100000:
        tries += 1
        o = outs[np.random.choice(len(outs), p=npairs/npairs.sum())]
        fs = bk[o]; a, b = random.sample(fs, 2); pr = tuple(sorted((a, b)))
        if pr in seen: continue
        seen.add(pr)
        pid = f"P{mi}{len(seen):03d}"
        both = (a in single) and (b in single)
        key.append(dict(pair_id=pid, method=m, form_a=pr[0], form_b=pr[1], output=o,
                        emo_a=single.get(pr[0], ""), emo_b=single.get(pr[1], ""),
                        labelled=both, x_by_emotion=(both and single[pr[0]] != single[pr[1]])))
        sheet.append(dict(pair_id=pid, form_a=pr[0], form_b=pr[1],
                          context_a1=ctx[pr[0]][0] if ctx[pr[0]] else "", context_a2=ctx[pr[0]][1] if len(ctx[pr[0]]) > 1 else "",
                          context_b1=ctx[pr[1]][0] if ctx[pr[1]] else "", context_b2=ctx[pr[1]][1] if len(ctx[pr[1]]) > 1 else "",
                          ann1="", ann2="", note=""))
    print(f"{m:<20} merged pairs {total:>8,}  labelled {lab:>6,}  coverage {stats[m]['coverage_pct']:6.2f}%  "
          f"X {x:>5,}  precision {prec:.4f}  sampled {len(seen)}")

sh_df = pd.DataFrame(sheet).sample(frac=1, random_state=42).reset_index(drop=True)
sh_df.to_csv("/kaggle/working/merge_audit_sheet.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(key).to_csv("/kaggle/working/merge_audit_key.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(stats.values()).to_csv("/kaggle/working/merge_coverage.csv", index=False)
print(f"\nwritten merge_audit_sheet.csv ({len(sh_df)} blind pairs, method hidden), merge_audit_key.csv, merge_coverage.csv")
print("Annotator instruction: ann1/ann2 = C (same word or same emotion), X (different word or different emotion), U (unclear).")
print("Give annotators the SHEET only; the key stays with you until scoring.")


single-emotion forms 170,767
keel                 merged pairs    1,475  labelled    969  coverage  65.69%  X     5  precision 0.9948  sampled 200
kope                 merged pairs    1,968  labelled    217  coverage  11.03%  X    27  precision 0.8756  sampled 200
metaphone            merged pairs   74,133  labelled  4,320  coverage   5.83%  X 2,094  precision 0.5153  sampled 200
terun                merged pairs   99,272  labelled  3,628  coverage   3.65%  X 1,570  precision 0.5673  sampled 200
soundex              merged pairs  342,835  labelled 15,248  coverage   4.45%  X 8,534  precision 0.4403  sampled 200
soundex_guard        merged pairs  273,543  labelled  9,647  coverage   3.53%  X 4,394  precision 0.5445  sampled 200
indicsoundex         merged pairs  338,498  labelled 15,125  coverage   4.47%  X 8,449  precision 0.4414  sampled 200
indicsoundex_guard   merged pairs  269,244  labelled  9,355  coverage   3.47%  X 4,195  precision 0.5516  sampled 200
lccn_base            merged

In [5]:
# ================================================================
#  CELL 2 - TABLE 5 COVERAGE COLUMN AND THE LABELLED-PAIR CORRELATION
# ================================================================
MEAN_F1 = {"keel": 0.7776, "kope": 0.7807, "metaphone": 0.7649, "terun": 0.7739, "soundex": 0.7536,
           "soundex_guard": 0.7578, "indicsoundex": 0.7530, "indicsoundex_guard": 0.7594,
           "lccn_base": 0.7783, "lccn": 0.7776}   # Table 4, mean over nine classifiers
cov = pd.DataFrame(stats.values()); cov["mean_macro_f1"] = cov.method.map(MEAN_F1)
print(cov.to_string(index=False))
r, pr_ = pearsonr(cov.precision_labelled, cov.mean_macro_f1); rho, ps = spearmanr(cov.precision_labelled, cov.mean_macro_f1)
print(f"\nlabelled-pair precision vs mean macro-F1 (10 methods): r = {r:.3f} (p = {pr_:.4f}), rho = {rho:.3f} (p = {ps:.4f})")
print("expect r = 0.881, rho = 0.839 from Section 4.2; coverage ranges from the Soundex family to keel")


            method  merged_pairs  labelled_pairs  coverage_pct  x_cross  precision_labelled  mean_macro_f1
              keel          1475             969         65.69        5              0.9948         0.7776
              kope          1968             217         11.03       27              0.8756         0.7807
         metaphone         74133            4320          5.83     2094              0.5153         0.7649
             terun         99272            3628          3.65     1570              0.5673         0.7739
           soundex        342835           15248          4.45     8534              0.4403         0.7536
     soundex_guard        273543            9647          3.53     4394              0.5445         0.7578
      indicsoundex        338498           15125          4.47     8449              0.4414         0.7530
indicsoundex_guard        269244            9355          3.47     4195              0.5516         0.7594
         lccn_base          3121     

In [6]:
# ================================================================
#  CELL 3 - PHASE 2: SCORE THE COMPLETED AUDIT (runs only if attached)
# ================================================================
done = glob.glob("/kaggle/input/**/merge_audit_done.csv", recursive=True)
if not done:
    print("merge_audit_done.csv not attached: Phase 2 skipped. Attach the completed sheet and re-run.")
else:
    a = pd.read_csv(done[0], dtype=str, keep_default_na=False); a.columns = [c.strip().lower() for c in a.columns]
    k = pd.read_csv("/kaggle/working/merge_audit_key.csv", dtype=str, keep_default_na=False)
    for c in ("ann1", "ann2"): a[c] = a[c].str.strip().str.upper().str[:1]
    a = a.merge(k[["pair_id", "method", "labelled", "x_by_emotion"]], on="pair_id", how="inner")
    v = a[a.ann1.isin(list("CXU")) & a.ann2.isin(list("CXU"))].copy()
    po = (v.ann1 == v.ann2).mean(); pe = sum((v.ann1 == c).mean() * (v.ann2 == c).mean() for c in "CXU")
    print(f"pairs {len(a)} | both annotated {len(v)} | Cohen's kappa {(po-pe)/(1-pe):.3f} (agreement {po:.3f})")
    gold = v[(v.ann1 == v.ann2) & (v.ann1 != "U")].copy(); gold["compatible"] = gold.ann1 == "C"
    v[v.ann1 != v.ann2].to_csv("/kaggle/working/merge_audit_disagreements.csv", index=False)
    print(f"adjudicated (agreement, not U): {len(gold)}; disagreements written to merge_audit_disagreements.csv")

    rows = []
    for m in METHODS:
        g = gold[gold.method == m]
        if not len(g): continue
        audited = g.compatible.mean()
        gl = g[g.labelled.astype(str).str.lower() == "true"]
        agree = ((gl.x_by_emotion.astype(str).str.lower() == "true") == (~gl.compatible)).mean() if len(gl) else float("nan")
        rows.append(dict(method=m, n_audited=len(g), precision_audited=round(audited, 4),
                         precision_labelled=stats[m]["precision_labelled"], coverage_pct=stats[m]["coverage_pct"],
                         n_labelled_in_sample=len(gl), human_vs_emotion_agreement=round(agree, 3) if len(gl) else None,
                         mean_macro_f1=MEAN_F1[m]))
    t13 = pd.DataFrame(rows); t13.to_csv("/kaggle/working/table13_precision_audit.csv", index=False)
    print("\n=== TABLE 13 ===\n", t13.to_string(index=False))
    r, pr_ = pearsonr(t13.precision_audited, t13.mean_macro_f1); rho, ps = spearmanr(t13.precision_audited, t13.mean_macro_f1)
    print(f"\naudited precision vs mean macro-F1: r = {r:.3f} (p = {pr_:.4f}), rho = {rho:.3f} (p = {ps:.4f})")
    print("compare with labelled-pair r = 0.881, rho = 0.839; ordering of methods:", list(t13.sort_values('precision_audited').method))
    print("\nDONE - table13_precision_audit.csv written; transcribe only from the committed log.")


merge_audit_done.csv not attached: Phase 2 skipped. Attach the completed sheet and re-run.
